Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 01 · Connect with a token and list what you may read

The first step of every client: connect to the Flight service with an API token that has the scope `data:read`, ask who the service thinks you are, and list what this token may read (kinds, BI models, measures, lake tables).

**You need:** Flight turned on (Admin → Engines → Arrow Flight), a token with `data:read` (My account → API tokens), the CA file of the server's certificate, Python 3.10+ with `pyarrow` and `pandas` (`requirements.txt`).

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## Connect

`connect()` opens a client for `DRISHTI_FLIGHT_URL`. The token is sent once, at the handshake; the service answers a session (`dfs_…`) that the client sends from then on. `whoami` answers the person, the person acted for (an application's token), the scopes and the limits of your role.

In [ ]:
f = connect()
who = f.whoami()
print("person:      ", who.get("person"))
print("on behalf of:", who.get("onBehalfOf") or "-")
print("scopes:      ", ", ".join(who.get("scopes", [])))
print("limits:      ", who.get("limits"))

## What this token may read

`catalogue()` is Flight's `ListFlights`: one request per kind, model, measure and lake table the token may read, names only (no data). Each entry is a request you can pass to `f.read(...)` as it is.

In [ ]:
import pandas as pd

def name(entry):
    r = entry.get("request")
    if isinstance(r, dict):
        return r.get("kind") or r.get("model") or r.get("measure") or r.get("lake") or str(r)
    return str(r)

catalogue = f.catalogue()
cat = pd.DataFrame({"form": [e.get("form") for e in catalogue], "name": [name(e) for e in catalogue]})
print(cat.groupby("form").size().to_string())
cat.head(20)

## The node you are talking to

`health` answers the node's name, the Arrow memory in use against its cap and the live reads. A probe holding a token can call it.

In [ ]:
f.health()

## What a refusal looks like

Every refusal carries Drishti's code (`DRS-56xx`, listed in the guide's troubleshooting table) and words that say what to do. A kind that does not exist is `DRS-5603`; a token without `data:read` would be `DRS-5602`.

In [ ]:
try:
    f.kind("no-such-kind")
except FlightRefused as e:
    print(e.code, "|", e.detail)

## Close

A client holds one connection per node it has read from; close it when done (or use `with connect() as f:`).

In [ ]:
f.close()